# Simulacion del Sistema de Parqueaderos - Supercentro

## Introduccion

Este cuaderno presenta la simulacion por eventos discretos del sistema de pago de parqueaderos del centro comercial Supercentro. El sistema cuenta con **tres cajeros independientes** en cada punto de salida, con filas independientes, sin cambio de fila ni desercion.

Se busca determinar si la cantidad actual de cajeros es suficiente o si se deben instalar mas para atender la demanda, considerando cuatro tipos de usuarios con diferentes velocidades de interaccion.

### Parametros del modelo

| Tipo de usuario | Proporcion | Tiempo de servicio (media) | Intervalo entre llegadas (media) |
|-----------------|------------|---------------------------|----------------------------------|
| Rapido          | 25%        | 1 minuto                  | 3 minutos                        |
| Normal          | 20%        | 3 minutos                 | 3 minutos                        |
| Lento           | 27,5%      | 4 minutos                 | 5 minutos                        |
| Muy lento       | 27,5%      | 6 minutos                 | 7 minutos                        |

### Configuracion de la simulacion

- **120 replicas** para el analisis de Welch (identificacion del transitorio)
- **40 replicas** independientes para las estadisticas finales
- **Horizonte**: 6.000 minutos por replica
- **Periodo de calentamiento**: 450 minutos (identificado por Welch)
- **Intervalos de confianza**: 95% con distribucion t de Student
- Cada cajero se modela como una cola **M/M/1** con disciplina **FCFS**

## Ejecucion de la simulacion

Se ejecuta el script de simulacion que genera los resultados, graficas y el archivo JSON con todas las estadisticas.

In [ ]:
import sys
sys.dont_write_bytecode = True
from pathlib import Path
import json

from simulacion_supercentro import run
resultados = run()
print("Simulacion completada exitosamente.")
print(f"Replicas ejecutadas: {resultados['configuration']['replications']}")
print(f"Horizonte: {resultados['configuration']['horizon']:.0f} minutos")
print(f"Periodo de calentamiento: {resultados['configuration']['warmup']:.0f} minutos")

---
## 1. Cajero con menor y mayor tiempo promedio de atencion

Se calculan las estadisticas de tiempo de servicio, espera y permanencia para cada cajero, junto con sus intervalos de confianza al 95%.

In [ ]:
print("="*70)
print("ESTADISTICAS POR CAJERO (configuracion actual: 3 cajeros)")
print("="*70)
print(f"{'Cajero':<8} {'Servicio (min)':<18} {'Espera (min)':<18} {'Permanencia (min)':<20} {'Utilizacion (%)':<18}")
print("-"*70)

for i, c in enumerate(resultados['cashiers']):
    srv = c['service']
    wait = c['wait']
    sys_t = c['system']
    util = c['utilization']
    print(f"{i+1:<8} {srv['mean']:.4f}             {wait['mean']:.2f}             {sys_t['mean']:.2f}               {util['mean']*100:.2f}")

min_c = resultados['minimum_observed_service_cashier']
max_c = resultados['maximum_observed_service_cashier']
print()
print(f"Cajero con MENOR tiempo de atencion: Cajero {min_c} ({resultados['cashiers'][min_c-1]['service']['mean']:.4f} min)")
print(f"Cajero con MAYOR tiempo de atencion: Cajero {max_c} ({resultados['cashiers'][max_c-1]['service']['mean']:.4f} min)")
print()
print("Nota: Los intervalos de confianza de los tres cajeros se superponen,")
print("lo que indica que la diferencia NO es estadisticamente significativa.")
print("Todos comparten la misma distribucion de servicio.")

In [ ]:
# Grafica de servicio y espera por cajero
from IPython.display import Image, display
display(Image(filename='figuras/cajeros.png', width=600))

**Figura 1.** Servicio y espera por cajero, con intervalos de confianza al 95%.

**Interpretacion:** Los tres cajeros presentan tiempos de servicio practicamente identicos (alrededor de 3,57 minutos). La espera media ronda los 14-15 minutos en cada cajero. Las barras de error se superponen, confirmando que no hay diferencia significativa entre ellos.

---

## 2. Promedio de usuarios de cada tipo en la totalidad de cajeros

Se contabilizan las llegadas de los cuatro tipos de usuario sumando los tres cajeros dentro de cada replica, y luego promediando las 40 replicas.

In [ ]:
tipos = ['Rapido', 'Normal', 'Lento', 'Muy lento']
arr = resultados['arrival_type_counts']
dep = resultados['departure_type_counts']
total_arr = sum(t['mean'] for t in arr)

print("="*70)
print("PROMEDIO DE USUARIOS POR TIPO (total de 3 cajeros, por replica)")
print("="*70)
print(f"{'Tipo':<12} {'Llegadas':<12} {'Proporcion':<12} {'Salidas':<12}")
print("-"*48)

for i, tipo in enumerate(tipos):
    prop = arr[i]['mean'] / total_arr * 100
    print(f"{tipo:<12} {arr[i]['mean']:.1f}        {prop:.2f}%       {dep[i]['mean']:.1f}")

print("-"*48)
print(f"{'Total':<12} {total_arr:.1f}        100.00%      {sum(t['mean'] for t in dep):.1f}")
print()
print(f"En promedio por replica llegan {total_arr:.0f} usuarios a los 3 cajeros.")
print("Las llegadas y salidas son practicamente iguales, confirmando")
print("que todos los usuarios son atendidos.")

In [ ]:
# Grafica de tipos de usuario
display(Image(filename='figuras/tipos_usuarios.png', width=600))

**Figura 2.** Composicion del flujo de llegada por tipo de usuario.

**Interpretacion:** Los usuarios lentos y muy lentos representan en conjunto el 55% del flujo (27,5% cada uno). Estos usuarios, con tiempos de servicio de 4 y 6 minutos respectivamente, son los que mas impactan la espera en las filas.

---

## 3. Estrategia de mejora

### Criterio de decision

Se selecciona el **menor numero de cajeros** cuyo limite superior del intervalo de confianza al 95% de la espera media sea **menor o igual a 5 minutos**, y cuyo semiancho relativo sea menor o igual al 10%.

### Escenarios evaluados

Se simularon configuraciones con 3, 4, 5 y 6 cajeros, manteniendo la demanda total constante.

In [ ]:
print("="*70)
print("COMPARACION DE ESCENARIOS")
print("="*70)
print(f"{'Cajeros':<10} {'Espera media':<15} {'IC 95%':<25} {'Precision':<12} {'Cumple'}")
print("-"*70)

for n in ['3', '4', '5', '6']:
    w = resultados['scenarios'][n]['wait']
    cumple = 'SI' if w['high'] <= 5.0 and w['relative_half_width'] <= 0.10 else 'NO'
    print(f"{n:<10} {w['mean']:.2f} min       [{w['low']:.2f} - {w['high']:.2f}] min   {w['relative_half_width']*100:.2f}%        {cumple}")

print()
print(f"Recomendacion: {resultados['recommended_cashiers']} cajeros por punto de salida.")

w3 = resultados['scenarios']['3']['wait']['mean']
w5 = resultados['scenarios']['5']['wait']['mean']
reduccion = (1 - w5/w3) * 100
print(f"Reduccion en espera media: {reduccion:.1f}% (de {w3:.2f} a {w5:.2f} minutos).")

In [ ]:
# Grafica de comparacion de escenarios
display(Image(filename='figuras/escenarios.png', width=600))

**Figura 3.** Comparacion de espera media por numero de cajeros.

**Interpretacion:** Con 3 cajeros la espera es inaceptable (14,63 min). Con 4 cajeros mejora pero no cumple el criterio (6,31 min). Con **5 cajeros** se logra una espera de 4,01 minutos, cumpliendo el criterio establecido. La configuracion de 6 cajeros es innecesariamente costosa.

**Conclusion:** Se recomienda instalar **5 cajeros** por punto de salida.

---

## 4. Verificacion, calibracion y validacion

### Verificacion
Se comprobo que el codigo implementa correctamente el modelo:
- Eventos procesados en orden cronologico
- Disciplina FCFS respetada en cada fila
- Sin solapamiento de servicios
- Esperas no negativas
- Conservacion: todo usuario que llega sale del sistema
- Utilizacion entre 0 y 1

### Calibracion (pendiente)
Requiere medir datos reales del centro comercial: tipos de usuario, intervalos de llegada, tiempos de servicio y variaciones horarias.

### Validacion (pendiente)
Requiere contrastar las salidas del modelo con una muestra independiente de datos reales.

In [ ]:
# Ejemplo de traza de verificacion: primeros 5 eventos
print("Traza de verificacion (primeros 5 eventos del cajero 1):")
print(f"{'Evento':<8} {'Llegada':<12} {'Inicio serv.':<14} {'Salida':<12} {'Espera':<10} {'Servicio':<10}")
print("-"*66)
for i, ev in enumerate(resultados['trace'][:5]):
    espera = ev['b'] - ev['a']
    print(f"{i+1:<8} {ev['a']:.2f}       {ev['b']:.2f}         {ev['d']:.2f}       {espera:.2f}      {ev['s']:.2f}")
print()
print("Se cumple: salida = inicio de servicio + duracion del servicio")

In [ ]:
# Diagrama del proceso de verificacion, calibracion y validacion
display(Image(filename='figuras/proceso_validacion.png', width=600))

**Figura 4.** Proceso de verificacion, calibracion y validacion del modelo.

---

## 5. Eliminacion del estado transitorio

Se utilizo la **tecnica de Welch** con 120 replicas independientes para identificar el periodo transitorio. Se muestreo la longitud de cola cada 10 minutos y se aplico un suavizado centrado con ventana de 15 muestras.

Se identifico un periodo de calentamiento de **450 minutos**, despues del cual el sistema alcanza el estado estable.

In [ ]:
# Grafica de Welch
display(Image(filename='figuras/welch.png', width=650))

**Figura 5.** Grafica de Welch: promedio suavizado de la longitud de cola (120 replicas).

**Interpretacion:** La linea se estabiliza alrededor de los 450 minutos, indicando que a partir de ese punto el sistema esta en estado estable. Los datos anteriores a ese punto corresponden al periodo transitorio.

### Comparacion antes y despues de eliminar el transitorio

In [ ]:
# Comparacion antes y despues
display(Image(filename='figuras/antes_despues.png', width=650))

**Figura 6.** Comparacion de los valores promedio antes y despues de eliminar el estado transitorio.

**Interpretacion:** Al eliminar los primeros 450 minutos (periodo transitorio), las estadisticas se calculan unicamente sobre el periodo en estado estable, dando mayor confiabilidad a los resultados.

In [ ]:
# Sensibilidad al periodo de calentamiento
print("Sensibilidad de la espera media al periodo de calentamiento:")
print(f"{'Calentamiento':<16} {'Espera media':<15} {'IC 95%':<25} {'Precision'}")
print("-"*65)
for w_str, vals in resultados['warmup_sensitivity'].items():
    print(f"{w_str + ' min':<16} {vals['mean']:.3f} min      [{vals['low']:.3f} - {vals['high']:.3f}]     {vals['relative_half_width']*100:.2f}%")
print()
print("Los resultados son estables con distintos periodos de calentamiento,")
print("confirmando la robustez de la eleccion de 450 minutos.")

In [ ]:
# Grafica de convergencia
display(Image(filename='figuras/convergencia.png', width=600))

**Figura 7.** Convergencia de la espera media a medida que aumentan las replicas.

**Interpretacion:** El intervalo de confianza se estrecha conforme se agregan replicas, confirmando que 40 replicas son suficientes para alcanzar la precision deseada (semiancho relativo menor al 10%).

---

## Conclusiones

1. **La configuracion actual de 3 cajeros es insuficiente.** La espera media es de 14,63 minutos con una utilizacion del 77%, lo que genera tiempos de espera excesivos para los usuarios.

2. **No hay diferencias significativas entre cajeros.** Los tres cajeros presentan tiempos de atencion practicamente identicos (alrededor de 3,57 minutos).

3. **La composicion de usuarios** sigue los parametros del modelo: 25% rapidos, 20% normales, 27,5% lentos y 27,5% muy lentos.

4. **Se recomienda instalar 5 cajeros por punto de salida.** Esta configuracion reduce la espera media a 4,01 minutos (una reduccion del 73%) y cumple el criterio de decision establecido.

5. **El transitorio se elimino correctamente** usando la tecnica de Welch, con un periodo de calentamiento de 450 minutos. Las 40 replicas garantizan la precision estadistica requerida.

6. **La verificacion interna fue satisfactoria.** La calibracion y validacion quedan pendientes de datos reales del centro comercial.